# Experiment 4 — Comparative Study of Deep CNN Architectures Using Transfer Learning
**CS3807 – Deep Learning Laboratory | Shiv Nadar University Chennai**

Run this notebook top-to-bottom on **Google Colab** with a **GPU runtime**
(`Runtime > Change runtime type > T4 GPU`). Total run time is roughly 10–15 minutes.

It covers, in order:
- Task 1: Dataset preparation (CIFAR-10)
- Task 2: Transfer learning setup (MobileNetV2 backbone, pretrained on ImageNet)
- Task 3: Model training
- Task 4: Fine-tuning
- Task 5: Evaluation + all 7 mandatory plots

Every figure is saved to disk as a `.png` so you can drop it straight into the report,
and every printed number below is what should go into the Results tables.

In [ ]:
# Basic setup
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.metrics import (confusion_matrix, classification_report,
                              precision_recall_fscore_support, ConfusionMatrixDisplay)
import time, os

os.makedirs('plots', exist_ok=True)
tf.random.set_seed(42)
np.random.seed(42)
print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

## Task 1: Dataset Preparation

CIFAR-10 has 50,000 training and 10,000 test images across 10 classes at 32×32×3.
We load it, normalize pixel values to [0,1], and inspect shapes and sample images.

In [ ]:
class_names = ['Airplane','Automobile','Bird','Cat','Deer',
               'Dog','Frog','Horse','Ship','Truck']

(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train = y_train.flatten()
y_test = y_test.flatten()

print("BEFORE preprocessing")
print("x_train:", x_train.shape, x_train.dtype)
print("x_test :", x_test.shape, x_test.dtype)

x_train_norm = x_train.astype('float32') / 255.0
x_test_norm  = x_test.astype('float32') / 255.0

print("\nAFTER normalization")
print("x_train:", x_train_norm.shape, "| min:", x_train_norm.min(), "max:", x_train_norm.max())
print("x_test :", x_test_norm.shape)

y_train_oh = keras.utils.to_categorical(y_train, 10)
y_test_oh  = keras.utils.to_categorical(y_test, 10)
print("y_train:", y_train_oh.shape, "(one-hot)")
print("y_test :", y_test_oh.shape, "(one-hot)")

In [ ]:
# Plot 1: Sample CIFAR-10 images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
rng = np.random.default_rng(42)
idxs = rng.choice(len(x_train), 10, replace=False)
for ax, idx in zip(axes.flat, idxs):
    ax.imshow(x_train[idx])
    ax.set_title(class_names[y_train[idx]], fontsize=10)
    ax.axis('off')
plt.suptitle('Plot 1: Sample CIFAR-10 Images')
plt.tight_layout()
plt.savefig('plots/01_sample_images.png', dpi=150)
plt.show()

## Task 2: Transfer Learning Setup

We use **MobileNetV2** pretrained on ImageNet as the backbone (a good CPU/Colab-friendly
choice; swap for `VGG16`, `ResNet50`, or `EfficientNetB0` by changing one line if your
manual requires a specific model). CIFAR-10 images are resized from 32×32 to 96×96 since
pretrained ImageNet backbones expect larger inputs. The convolutional base is frozen, and
a new classification head is added on top.

In [ ]:
IMG_SIZE = 96
BATCH_SIZE = 64

def make_dataset(x, y, training=True):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(10000, seed=42)
    def _prep(img, label):
        img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
        img = preprocess_input(img * 255.0)
        return img, label
    ds = ds.map(_prep, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

# 90/10 train/validation split
n_val = 5000
x_val, y_val = x_train_norm[:n_val], y_train_oh[:n_val]
x_tr,  y_tr  = x_train_norm[n_val:], y_train_oh[n_val:]

train_ds = make_dataset(x_tr, y_tr, training=True)
val_ds   = make_dataset(x_val, y_val, training=False)
test_ds  = make_dataset(x_test_norm, y_test_oh, training=False)

In [ ]:
base_model = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3),
                          include_top=False,
                          weights='imagenet')
base_model.trainable = False  # freeze convolutional base

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(10, activation='softmax')(x)

model = keras.Model(inputs, outputs)
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

## Task 3: Model Training

Frozen-base training for 10 epochs (Adam, batch size 32/64, categorical cross-entropy),
as specified in the lab manual.

In [ ]:
start = time.time()
history = model.fit(train_ds, validation_data=val_ds, epochs=10)
base_train_time = time.time() - start
print(f"Base training time: {base_train_time:.2f} seconds")

## Task 4: Fine Tuning

Unfreeze the last convolutional block of MobileNetV2 and continue training at a lower
learning rate for a few more epochs.

In [ ]:
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 30  # unfreeze last ~30 layers
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

start = time.time()
history_ft = model.fit(train_ds, validation_data=val_ds, epochs=8)
fine_tune_time = time.time() - start
print(f"Fine-tuning time: {fine_tune_time:.2f} seconds")

# Combine histories for plotting
def combine(h1, h2, key):
    return h1.history[key] + h2.history[key]

train_acc = combine(history, history_ft, 'accuracy')
val_acc   = combine(history, history_ft, 'val_accuracy')
train_loss = combine(history, history_ft, 'loss')
val_loss   = combine(history, history_ft, 'val_loss')

In [ ]:
# Plots 2 & 3: Training vs Validation Accuracy / Loss
epochs_range = range(1, len(train_acc) + 1)

plt.figure(figsize=(8,5))
plt.plot(epochs_range, train_acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.axvline(x=10.5, color='gray', linestyle='--', label='Fine-tuning starts')
plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy'); plt.legend()
plt.tight_layout(); plt.savefig('plots/02_accuracy.png', dpi=150); plt.show()

plt.figure(figsize=(8,5))
plt.plot(epochs_range, train_loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.axvline(x=10.5, color='gray', linestyle='--', label='Fine-tuning starts')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Training vs Validation Loss'); plt.legend()
plt.tight_layout(); plt.savefig('plots/03_loss.png', dpi=150); plt.show()

## Task 5: Model Evaluation

Accuracy, precision, recall, F1-score, confusion matrix and classification report on the
held-out CIFAR-10 test set.

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test Accuracy: {test_acc:.4f}")

y_pred_probs = model.predict(test_ds)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = y_test

precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro')
print(f"Precision (macro): {precision:.4f}")
print(f"Recall (macro):    {recall:.4f}")
print(f"F1-score (macro):  {f1:.4f}")

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))

total_params = model.count_params()
print(f"Total parameters: {total_params:,}")

In [ ]:
# Plot: Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8,8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, cmap='Blues', xticks_rotation=45, colorbar=True)
plt.title('Confusion Matrix (Test Set)')
plt.tight_layout(); plt.savefig('plots/04_confusion_matrix.png', dpi=150); plt.show()

In [ ]:
# Optional: Misclassified images
mis_idx = np.where(y_pred != y_true)[0]
sample_mis = np.random.default_rng(0).choice(mis_idx, min(10, len(mis_idx)), replace=False)

fig, axes = plt.subplots(2, 5, figsize=(12,5))
for ax, idx in zip(axes.flat, sample_mis):
    ax.imshow(x_test[idx])
    ax.set_title(f"True: {class_names[y_true[idx]]}\nPred: {class_names[y_pred[idx]]}", fontsize=9)
    ax.axis('off')
plt.suptitle('Misclassified Test Images')
plt.tight_layout(); plt.savefig('plots/05_misclassified.png', dpi=150); plt.show()

## Summary table (copy these numbers into the report)

In [ ]:
import pandas as pd
summary = pd.DataFrame({
    'Metric': ['Training Accuracy (final)', 'Validation Accuracy (final)', 'Test Accuracy',
               'Precision (macro)', 'Recall (macro)', 'F1-score (macro)',
               'Base Training Time (s)', 'Fine-tune Time (s)', 'Total Parameters'],
    'Value': [f"{train_acc[-1]:.4f}", f"{val_acc[-1]:.4f}", f"{test_acc:.4f}",
              f"{precision:.4f}", f"{recall:.4f}", f"{f1:.4f}",
              f"{base_train_time:.2f}", f"{fine_tune_time:.2f}", f"{total_params:,}"]
})
summary

---
### Notes
- To switch backbones (e.g. to match a manual that mandates VGG16 or ResNet50), replace
  the `MobileNetV2` import and constructor in Task 2 with `VGG16` / `ResNet50` from
  `tensorflow.keras.applications` — the rest of the pipeline is unchanged.
- All plots are saved under `plots/` for direct use in the Word report.
- Re-run Task 4 with more/fewer unfrozen layers or a different learning rate for the
  hyperparameter study in Section 16 of the manual.